# 06 — Equity Decomposition, Multi-Destination Analysis, and Model Validation

**Purpose**: Deepens and validates the Bayesian posterior analysis from nb04/nb05 before the intervention simulation (nb07).  
Covers: formal equity metrics (Gini, Concentration Index), subgroup posterior analysis, multi-destination equity,  
composite deficit index, posterior predictive checks, LOO-CV model comparison, prior sensitivity, Moran's I for raw X, drop-one robustness.

**Previous:** [`05_posterior_analysis.ipynb`](05_posterior_analysis.ipynb)  
**Next:** [`07_intervention_simulation.ipynb`](07_intervention_simulation.ipynb)

| Section | Content |
|---|---|
| §1 | Environment setup and canonical data load |
| §2 | Formal equity metrics — Gini coefficient + Concentration Index |
| §3 | Subgroup posterior analysis (quartiles, vehicle ownership, agency) |
| §4 | Multi-destination equity analysis (jobs / hospitals / groceries / schools) |
| §5 | Composite accessibility deficit index |
| §6 | Posterior predictive checks (PPCs) |
| §7 | LOO-CV model comparison (raw X vs Spatial+) |
| §8 | Prior sensitivity grid (importance weighting) |
| §9 | Moran's I for raw X estimand — closes F026 |
| §10 | Drop-one covariate robustness |
| §11 | Summary export |


## §1 — Environment Setup and Canonical Data Load

In [ ]:
from __future__ import annotations
import os, warnings
from pathlib import Path
import yaml
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy import stats
from scipy.stats import spearmanr, rankdata
from scipy.special import logsumexp
warnings.filterwarnings('ignore', category=FutureWarning)

def find_repo_root() -> Path:
    start = Path.cwd().resolve()
    for d in [start, *start.parents]:
        if (d / 'configs' / 'san_diego.yaml').exists():
            return d
    raise FileNotFoundError('Could not find configs/san_diego.yaml')

REPO_ROOT = find_repo_root()
with open(REPO_ROOT / 'configs' / 'san_diego.yaml') as f:
    CITY = yaml.safe_load(f)
with open(REPO_ROOT / 'configs' / 'defaults.yaml') as f:
    DEFAULTS = yaml.safe_load(f)

print(f'REPO_ROOT = {REPO_ROOT}')
print(f'city = {CITY.get("city_label") or CITY.get("city")}')

In [ ]:
# ── Config pin ──────────────────────────────────────────────────────────────
RID             = os.environ.get('PIPELINE_RUN_ID', 'fit_raw_zscore_x')
THRESHOLD_MIN   = int(os.environ.get('THRESHOLD_MIN', 45))
DESERT_QUANTILE = float(os.environ.get('DESERT_QUANTILE', 0.25))
N_DRAWS_SAMPLE  = int(os.environ.get('N_DRAWS_SAMPLE', 1000))  # draws for uncertainty propagation

ARTIFACT_TABLES  = REPO_ROOT / 'artifacts' / 'tables'
ARTIFACT_FIGURES = REPO_ROOT / 'artifacts' / 'figures'
ARTIFACT_TABLES.mkdir(parents=True, exist_ok=True)
ARTIFACT_FIGURES.mkdir(parents=True, exist_ok=True)

def artifact_table(stem: str) -> Path:
    return ARTIFACT_TABLES / f'pipeline__06_{stem}__{RID}.csv'

def artifact_figure(stem: str) -> Path:
    return ARTIFACT_FIGURES / f'pipeline__06_{stem}__{RID}.png'

print(f'RID={RID}  threshold={THRESHOLD_MIN}min  desert_q={DESERT_QUANTILE}  n_draws={N_DRAWS_SAMPLE}')

In [ ]:
# ── Load posterior summary ───────────────────────────────────────────────────
post_summary_path = REPO_ROOT / 'data' / 'processed' / 'posteriors' / f'{RID}_posterior_summary.parquet'
posterior_summary = pd.read_parquet(post_summary_path)
print(f'Posterior summary: {posterior_summary.shape}  columns: {list(posterior_summary.columns[:8])}...')

In [ ]:
# ── Load posterior samples (N_DRAWS_SAMPLE random rows for tractable computation) ──
post_samples_path = REPO_ROOT / 'data' / 'processed' / 'posteriors' / f'{RID}_posterior_samples.parquet'
all_samples = pd.read_parquet(post_samples_path)
print(f'Full samples: {all_samples.shape}')

# Pivot to wide format: rows=tracts, cols=draws
# Expected columns: GEOID, draw, mu (or posterior_jobs)
sample_cols = list(all_samples.columns)
print(f'Sample columns: {sample_cols[:10]}')

# Detect draw and value columns
DRAW_COL  = 'draw' if 'draw' in sample_cols else 'sample'
# mu column — prefer 'mu', fallback to 'posterior_jobs' or 'y_hat'
for candidate in ('mu', 'posterior_jobs', 'y_hat', 'mu_back'):
    if candidate in sample_cols:
        MU_COL = candidate
        break
else:
    MU_COL = sample_cols[-1]  # last numeric column as fallback

print(f'Using draw_col={DRAW_COL!r}, mu_col={MU_COL!r}')

# Pivot: shape (n_tracts, n_draws)
samples_wide = all_samples.pivot(index='GEOID', columns=DRAW_COL, values=MU_COL)
print(f'Samples wide: {samples_wide.shape}  (tracts × draws)')

# Subsample draws for heavy computations
rng = np.random.default_rng(42)
if samples_wide.shape[1] > N_DRAWS_SAMPLE:
    draw_idx = rng.choice(samples_wide.shape[1], N_DRAWS_SAMPLE, replace=False)
    samples_sub = samples_wide.iloc[:, draw_idx]
else:
    samples_sub = samples_wide
print(f'Subsampled draws: {samples_sub.shape[1]}')

In [ ]:
# ── Load accessibility bundle ────────────────────────────────────────────────
acc_path = sorted((REPO_ROOT / 'data' / 'processed' / 'accessibility').glob('tract_accessibility_bundle__*.parquet'))[-1]
acc_bundle = pd.read_parquet(acc_path)
acc_bundle['GEOID'] = acc_bundle['GEOID'].astype(str).str.zfill(11)
print(f'Accessibility bundle: {acc_bundle.shape}  from {acc_path.name}')
print('Columns:', list(acc_bundle.columns))

In [ ]:
# ── Load ACS tract attributes ────────────────────────────────────────────────
acs_candidates = sorted((REPO_ROOT / 'artifacts' / 'tables').glob('eda__acs_sd_tract_attributes__*.csv'))
if not acs_candidates:
    acs_candidates = sorted((REPO_ROOT / 'data' / 'processed').glob('*acs*tract*.csv'))
acs_path = acs_candidates[-1]
acs = pd.read_csv(acs_path, dtype={'GEOID': str})
acs['GEOID'] = acs['GEOID'].str.zfill(11)
print(f'ACS: {acs.shape}  from {acs_path.name}')
print('Disadvantage col:', 'disadvantage_z' if 'disadvantage_z' in acs.columns else [c for c in acs.columns if 'disadv' in c.lower()])

In [ ]:
# ── Load tract geometries ────────────────────────────────────────────────────
tiger_candidates = sorted((REPO_ROOT / 'data' / 'raw' / 'census').glob('tl_*_06_tract*.shp'))
if not tiger_candidates:
    tiger_candidates = sorted((REPO_ROOT / 'data' / 'raw' / 'census').glob('*.shp'))
tracts_gdf = gpd.read_file(tiger_candidates[-1])
tracts_gdf['GEOID'] = tracts_gdf['GEOID'].astype(str).str.zfill(11)
print(f'Tracts shapefile: {tracts_gdf.shape}  CRS: {tracts_gdf.crs}')

In [ ]:
# ── Build master analysis frame ──────────────────────────────────────────────
# Align posterior_summary with ACS and accessibility on GEOID
posterior_summary['GEOID'] = posterior_summary['GEOID'].astype(str).str.zfill(11)

master = (
    posterior_summary
    .merge(acs[['GEOID','disadvantage_z','no_vehicle_hh_rate','log_median_income',
                'log_pop_density','poverty_rate','total_pop']].copy(),
           on='GEOID', how='inner')
    .merge(acc_bundle, on='GEOID', how='inner')
)

# Merge spatial
master_geo = tracts_gdf[['GEOID','geometry']].merge(master, on='GEOID', how='inner')
master_geo = gpd.GeoDataFrame(master_geo, geometry='geometry', crs=tracts_gdf.crs)

# Detect key posterior columns
MEAN_COL  = next((c for c in master.columns if 'mean' in c.lower() and 'job' in c.lower()), None)
SD_COL    = next((c for c in master.columns if 'sd' in c.lower() and 'log' in c.lower()), None)
EXCEED_COL = next((c for c in master.columns if 'exceedance' in c.lower() or 'p_low' in c.lower()), None)

print(f'Master: {master.shape}')
print(f'  mean_col={MEAN_COL!r}  sd_col={SD_COL!r}  exceed_col={EXCEED_COL!r}')

# Fallback: use samples_wide row means if posterior_summary lacks a jobs mean column
if MEAN_COL is None:
    samples_wide_aligned = samples_sub.loc[samples_sub.index.isin(master['GEOID'])]
    master = master.set_index('GEOID')
    master['posterior_mean_jobs'] = samples_wide_aligned.mean(axis=1)
    master = master.reset_index()
    MEAN_COL = 'posterior_mean_jobs'
    print(f'  computed MEAN_COL from draws: {MEAN_COL}')

print(f'Master geometry: {master_geo.shape}')

## §2 — Formal Equity Metrics: Gini Coefficient + Concentration Index

Standard distributional inequality metrics applied to posterior job accessibility.  
Uncertainty is propagated by computing each metric across N posterior draws → posterior CI on the metric itself.

In [ ]:
def gini_weighted(values: np.ndarray, weights: np.ndarray | None = None) -> float:
    """Population-weighted Gini coefficient. Values must be >= 0."""
    values = np.asarray(values, dtype=float)
    if weights is None:
        weights = np.ones_like(values)
    weights = np.asarray(weights, dtype=float)
    # Sort by value
    idx = np.argsort(values)
    v, w = values[idx], weights[idx]
    w_cum = np.cumsum(w)
    # Standard Gini formula
    total_w = w_cum[-1]
    total_wv = np.sum(w * v)
    gini = (2 * np.sum(w * v * (w_cum - w / 2)) / (total_w * total_wv)) - 1
    return float(gini)


def concentration_index(y: np.ndarray, rank_var: np.ndarray) -> float:
    """Health-economics Concentration Index (Wagstaff 2005).
    Positive => concentrated among high-rank (disadvantaged) tracts.
    rank_var should be disadvantage_z so high rank = high disadvantage.
    """
    n = len(y)
    r = rankdata(rank_var) / n  # fractional rank
    mu = np.mean(y)
    if mu == 0:
        return 0.0
    return float(2 * np.cov(y, r)[0, 1] / mu)


print('Gini and Concentration Index functions defined.')

In [ ]:
# ── Point-estimate equity metrics ────────────────────────────────────────────
pop = master['total_pop'].fillna(1).values.clip(1)
mean_jobs = master[MEAN_COL].values
det_jobs  = master[f'jobs_C000_{THRESHOLD_MIN}min'].values
disadv    = master['disadvantage_z'].values

# Shift to zero-baseline (Gini requires non-negative)
mean_jobs_pos = mean_jobs - mean_jobs.min() + 1e-6
det_jobs_pos  = det_jobs - det_jobs.min() + 1e-6

gini_post = gini_weighted(mean_jobs_pos, pop)
gini_det  = gini_weighted(det_jobs_pos, pop)
ci_post   = concentration_index(mean_jobs, disadv)
ci_det    = concentration_index(det_jobs,  disadv)

if EXCEED_COL:
    exceed_vals = master[EXCEED_COL].values
    ci_exceed   = concentration_index(exceed_vals, disadv)
    gini_exceed = gini_weighted(exceed_vals.clip(0), pop)
else:
    ci_exceed = gini_exceed = float('nan')

print(f'Gini (posterior mean jobs): {gini_post:.4f}')
print(f'Gini (deterministic jobs):  {gini_det:.4f}')
print(f'Concentration Index (posterior mean): {ci_post:.4f}  [pos=pro-poor]')
print(f'Concentration Index (deterministic):  {ci_det:.4f}')
print(f'Concentration Index (exceedance):     {ci_exceed:.4f}')

In [ ]:
# ── Propagate uncertainty through equity metrics ─────────────────────────────
# For each posterior draw: compute Gini and CI → posterior distribution of metric
geoid_order = master['GEOID'].values
pop_aligned = pop.copy()

# Align samples to master row order
common_geoids = [g for g in geoid_order if g in samples_sub.index]
samples_aligned = samples_sub.loc[common_geoids].values  # (n_tracts, n_draws)
pop_aligned_sub = master.set_index('GEOID').loc[common_geoids, 'total_pop'].fillna(1).values.clip(1)
disadv_sub      = master.set_index('GEOID').loc[common_geoids, 'disadvantage_z'].values

gini_draws = []
ci_draws   = []
for i in range(samples_aligned.shape[1]):
    draw_vals = samples_aligned[:, i]
    draw_pos  = draw_vals - draw_vals.min() + 1e-6
    gini_draws.append(gini_weighted(draw_pos, pop_aligned_sub))
    ci_draws.append(concentration_index(draw_vals, disadv_sub))

gini_draws = np.array(gini_draws)
ci_draws   = np.array(ci_draws)

print(f'Posterior Gini: mean={gini_draws.mean():.4f}  95% CI [{np.percentile(gini_draws,2.5):.4f}, {np.percentile(gini_draws,97.5):.4f}]')
print(f'Posterior CI:   mean={ci_draws.mean():.4f}  95% CI [{np.percentile(ci_draws,2.5):.4f}, {np.percentile(ci_draws,97.5):.4f}]')

In [ ]:
# ── Lorenz / Concentration curve figure ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ── Left: Lorenz curve (inequality in access) ───────────────────────────────
ax = axes[0]
idx_sort = np.argsort(mean_jobs)
cumshare_pop = np.cumsum(pop[idx_sort]) / pop.sum()
cumshare_acc = np.cumsum(mean_jobs_pos[idx_sort]) / mean_jobs_pos.sum()
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Perfect equality')
ax.plot(cumshare_pop, cumshare_acc, color='steelblue', lw=2, label=f'Posterior mean (Gini={gini_post:.3f})')
idx_sort_det = np.argsort(det_jobs)
cumshare_acc_det = np.cumsum(det_jobs_pos[idx_sort_det]) / det_jobs_pos.sum()
cumshare_pop_det = np.cumsum(pop[idx_sort_det]) / pop.sum()
ax.plot(cumshare_pop_det, cumshare_acc_det, color='coral', lw=1.5, ls='--', label=f'Deterministic (Gini={gini_det:.3f})')
ax.fill_between(cumshare_pop, cumshare_pop, cumshare_acc, alpha=0.15, color='steelblue')
ax.set_xlabel('Cumulative population share (sorted by access)')
ax.set_ylabel('Cumulative job accessibility share')
ax.set_title('Lorenz Curve — Job Accessibility')
ax.legend(fontsize=9)
ax.set_aspect('equal')

# ── Right: Concentration curve (access vs disadvantage rank) ────────────────
ax2 = axes[1]
idx_disadv = np.argsort(disadv)  # low → high disadvantage
cum_disadv_pop  = np.cumsum(pop[idx_disadv]) / pop.sum()
cum_disadv_acc  = np.cumsum(mean_jobs_pos[idx_disadv]) / mean_jobs_pos.sum()
ax2.plot([0, 1], [0, 1], 'k--', lw=1, label='Perfect equality')
ax2.plot(cum_disadv_pop, cum_disadv_acc, color='mediumseagreen', lw=2, label=f'Posterior mean (CI={ci_post:.3f})')
ax2.fill_between(cum_disadv_pop, cum_disadv_pop, cum_disadv_acc,
                 where=cum_disadv_acc > cum_disadv_pop, alpha=0.15, color='mediumseagreen', label='Pro-poor zone')
ax2.fill_between(cum_disadv_pop, cum_disadv_pop, cum_disadv_acc,
                 where=cum_disadv_acc <= cum_disadv_pop, alpha=0.15, color='coral', label='Pro-rich zone')
ax2.set_xlabel('Cumulative population share (sorted by disadvantage, low→high)')
ax2.set_ylabel('Cumulative job accessibility share')
ax2.set_title('Concentration Curve — Accessibility vs Disadvantage')
ax2.legend(fontsize=9)
ax2.set_aspect('equal')

plt.tight_layout()
fig_path = artifact_figure('lorenz_curve')
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

In [ ]:
# ── Export equity metrics CSV ────────────────────────────────────────────────
equity_rows = [
    dict(metric='Gini_posterior_mean_jobs', point=gini_post, posterior_mean=gini_draws.mean(),
         ci_lower=np.percentile(gini_draws, 2.5), ci_upper=np.percentile(gini_draws, 97.5),
         note='Population-weighted; 0=perfect equality, 1=maximum inequality'),
    dict(metric='Gini_deterministic_jobs',  point=gini_det, posterior_mean=float('nan'),
         ci_lower=float('nan'), ci_upper=float('nan'), note='Deterministic baseline'),
    dict(metric='CI_posterior_mean_jobs',   point=ci_post, posterior_mean=ci_draws.mean(),
         ci_lower=np.percentile(ci_draws, 2.5), ci_upper=np.percentile(ci_draws, 97.5),
         note='Concentration Index vs disadvantage_z; positive=pro-poor'),
    dict(metric='CI_deterministic_jobs',    point=ci_det, posterior_mean=float('nan'),
         ci_lower=float('nan'), ci_upper=float('nan'), note='Deterministic CI baseline'),
    dict(metric='CI_exceedance_prob',       point=ci_exceed, posterior_mean=float('nan'),
         ci_lower=float('nan'), ci_upper=float('nan'), note='CI on desert exceedance vs disadvantage'),
]
equity_df = pd.DataFrame(equity_rows)
p = artifact_table('gini_ci_equity')
equity_df.to_csv(p, index=False)
print(equity_df.to_string())
print(f'Saved: {p}')

## §3 — Subgroup Posterior Analysis

Disaggregate beyond aggregate Spearman. Groups: disadvantage quartiles, vehicle ownership, agency service area.  
Key output: posterior uncertainty of the equity gap (Q4 median − Q1 median).

In [ ]:
# ── Define groups ────────────────────────────────────────────────────────────
df = master.copy()

# Disadvantage quartiles
df['disadv_q'] = pd.qcut(df['disadvantage_z'], q=4, labels=['Q1 (least)','Q2','Q3','Q4 (most)'])

# Vehicle ownership: high dependency = no_vehicle_hh_rate > 0.30
df['veh_group'] = np.where(df['no_vehicle_hh_rate'] > 0.30, 'High transit-dependency (>30% no-car)', 'Lower transit-dependency')

# Poverty quartiles
if 'poverty_rate' in df.columns:
    df['poverty_q'] = pd.qcut(df['poverty_rate'], q=4, labels=['P1 (lowest)','P2','P3','P4 (highest)'])

# Agency area: detect from accessibility bundle stop counts if available
# Proxy: if MTS stops >> 0 and NCTD stops == 0 → MTS area; vice versa; both > 0 → mixed
if 'mts_stop_count' in df.columns and 'nctd_stop_count' in df.columns:
    df['agency_area'] = 'MTS-only'
    df.loc[(df['nctd_stop_count'] > 0) & (df['mts_stop_count'] == 0), 'agency_area'] = 'NCTD-only'
    df.loc[(df['nctd_stop_count'] > 0) & (df['mts_stop_count'] > 0), 'agency_area'] = 'Both agencies'
else:
    df['agency_area'] = 'All tracts (agency data unavailable)'  

print('Groups defined.')
print(df['disadv_q'].value_counts().sort_index())

In [ ]:
# ── Subgroup statistics with posterior uncertainty ───────────────────────────
def subgroup_stats(df_in, group_col, samples_wide_df):
    """For each group level: posterior median, 95% CI, exceedance fraction.
    Also compute posterior distribution of the gap between top and bottom group.
    """
    rows = []
    group_draw_medians = {}  # group → array of per-draw medians
    
    for grp_name, grp_df in df_in.groupby(group_col, observed=True):
        geoids = grp_df['GEOID'].values
        # Subset samples
        common = [g for g in geoids if g in samples_wide_df.index]
        if len(common) < 5:
            continue
        grp_samples = samples_wide_df.loc[common].values  # (n_tracts_in_group, n_draws)
        
        # Per-draw group median
        per_draw_medians = np.median(grp_samples, axis=0)
        group_draw_medians[str(grp_name)] = per_draw_medians
        
        post_mean_pt = grp_df[MEAN_COL].mean()
        exceed_frac  = grp_df[EXCEED_COL].gt(0.5).mean() if EXCEED_COL else float('nan')
        
        rows.append(dict(
            group=str(grp_name),
            n_tracts=len(common),
            posterior_mean_pt=post_mean_pt,
            posterior_median_of_group_median=float(np.median(per_draw_medians)),
            ci_lower=float(np.percentile(per_draw_medians, 2.5)),
            ci_upper=float(np.percentile(per_draw_medians, 97.5)),
            frac_exceedance_gt50=float(exceed_frac),
        ))
    
    stats_df = pd.DataFrame(rows).sort_values('posterior_median_of_group_median', ascending=False)
    return stats_df, group_draw_medians


print('Running subgroup analysis for disadvantage quartiles...')
disadv_stats, disadv_draws = subgroup_stats(df, 'disadv_q', samples_sub)
print(disadv_stats.to_string(index=False))

In [ ]:
print('Running subgroup analysis for vehicle ownership...')
veh_stats, veh_draws = subgroup_stats(df, 'veh_group', samples_sub)
print(veh_stats.to_string(index=False))

if df['agency_area'].nunique() > 1:
    print('\nRunning subgroup analysis for agency area...')
    agency_stats, agency_draws = subgroup_stats(df, 'agency_area', samples_sub)
    print(agency_stats.to_string(index=False))
else:
    agency_stats = pd.DataFrame()

# Posterior probability Q4 > Q1
if 'Q1 (least)' in disadv_draws and 'Q4 (most)' in disadv_draws:
    p_q4_gt_q1 = np.mean(disadv_draws['Q4 (most)'] > disadv_draws['Q1 (least)'])
    gap_median  = np.median(disadv_draws['Q4 (most)'] - disadv_draws['Q1 (least)'])
    gap_ci      = np.percentile(disadv_draws['Q4 (most)'] - disadv_draws['Q1 (least)'], [2.5, 97.5])
    print(f'\nP(Q4 median jobs > Q1 median jobs) = {p_q4_gt_q1:.3f}')
    print(f'Gap (Q4 − Q1) posterior median: {gap_median:.2f}  95% CI [{gap_ci[0]:.2f}, {gap_ci[1]:.2f}]')

In [ ]:
# ── Forest plot figure ───────────────────────────────────────────────────────
all_subgroup_dfs = []
for label, sdf in [('Disadvantage quartile', disadv_stats),
                   ('Vehicle ownership', veh_stats),
                   ('Agency area', agency_stats)]:
    if not sdf.empty:
        sdf2 = sdf.copy()
        sdf2['group_type'] = label
        all_subgroup_dfs.append(sdf2)

all_sg = pd.concat(all_subgroup_dfs, ignore_index=True)

fig, ax = plt.subplots(figsize=(10, max(6, len(all_sg) * 0.5)))
y_pos = np.arange(len(all_sg))
ax.errorbar(
    x=all_sg['posterior_median_of_group_median'],
    y=y_pos,
    xerr=[all_sg['posterior_median_of_group_median'] - all_sg['ci_lower'],
          all_sg['ci_upper'] - all_sg['posterior_median_of_group_median']],
    fmt='o', color='steelblue', ecolor='steelblue', capsize=4, ms=7, lw=1.8
)
ax.set_yticks(y_pos)
labels_fmt = [f"{r.group}  (n={r.n_tracts})" for _, r in all_sg.iterrows()]
ax.set_yticklabels(labels_fmt, fontsize=9)
ax.set_xlabel('Posterior median of group-median job accessibility')
ax.set_title('Subgroup Posterior Analysis — Job Accessibility\n95% Credible Intervals')
ax.axvline(all_sg['posterior_median_of_group_median'].median(), color='gray', ls=':', lw=1)
ax.invert_yaxis()
plt.tight_layout()
fig_path = artifact_figure('subgroup_forest_plot')
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

p = artifact_table('subgroup_posterior_summary')
all_sg.to_csv(p, index=False)
print(f'Saved: {p}')

## §4 — Multi-Destination Equity Analysis

Does the pro-poor accessibility gradient hold for hospitals, groceries, and schools — not just jobs?  
Spearman ρ × 4 destinations × 3 thresholds. Identify multi-destination deserts.

In [ ]:
# ── Destination column map ───────────────────────────────────────────────────
DEST_MAP = {
    'Jobs (LODES)':  {'30': 'jobs_C000_30min', '45': 'jobs_C000_45min', '60': 'jobs_C000_60min'},
    'Hospitals':     {'30': 'hospitals_count_30min', '45': 'hospitals_count_45min', '60': 'hospitals_count_60min'},
    'Groceries':     {'30': 'groceries_count_30min', '45': 'groceries_count_45min', '60': 'groceries_count_60min'},
    'Schools':       {'30': 'schools_count_30min',  '45': 'schools_count_45min',  '60': 'schools_count_60min'},
}

# Verify columns exist
for dest, cols in DEST_MAP.items():
    for t, col in cols.items():
        exists = col in master.columns
        if not exists:
            print(f'  MISSING: {col}')

print('Destination column check done.')

In [ ]:
# ── Spearman table: destination × threshold ──────────────────────────────────
multidest_rows = []
for dest_name, col_map in DEST_MAP.items():
    for threshold in ['30', '45', '60']:
        col = col_map.get(threshold)
        if col not in master.columns:
            continue
        vals = master[col].fillna(0).values
        rho, p = spearmanr(vals, disadv)
        multidest_rows.append(dict(
            destination=dest_name,
            threshold_min=int(threshold),
            col=col,
            n_tracts=len(vals),
            spearman_rho=round(rho, 4),
            p_value=float(f'{p:.3e}'),
            direction='pro-poor' if rho > 0.05 else ('anti-poor' if rho < -0.05 else 'neutral'),
        ))

multidest_df = pd.DataFrame(multidest_rows)
print(multidest_df.pivot(index='destination', columns='threshold_min', values='spearman_rho').to_string())

p = artifact_table('multidestination_equity')
multidest_df.to_csv(p, index=False)
print(f'\nSaved: {p}')

In [ ]:
# ── Heatmap: destination × threshold × Spearman ρ ───────────────────────────
pivot = multidest_df.pivot(index='destination', columns='threshold_min', values='spearman_rho')

fig, ax = plt.subplots(figsize=(7, 4))
im = ax.imshow(pivot.values, cmap='RdYlGn', vmin=-0.6, vmax=0.6, aspect='auto')
plt.colorbar(im, ax=ax, label='Spearman ρ (vs disadvantage_z)')
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([f'{c} min' for c in pivot.columns])
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
ax.set_title('Equity Gradient by Destination Type and Travel-Time Threshold\n'
             'Green = pro-poor (advantaged by transit), Red = anti-poor')
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.iloc[i, j]
        ax.text(j, i, f'{val:+.3f}', ha='center', va='center', fontsize=11,
                color='black' if abs(val) < 0.4 else 'white')
plt.tight_layout()
fig_path = artifact_figure('multidestination_spearman_heatmap')
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

In [ ]:
# ── Multi-destination deserts (≥3 of 4 destinations below Q25) ───────────────
desert_flags = pd.DataFrame({'GEOID': master['GEOID']})
for dest_name, col_map in DEST_MAP.items():
    col = col_map['45']
    if col in master.columns:
        q25 = master[col].quantile(DESERT_QUANTILE)
        desert_flags[f'{dest_name}_desert'] = (master[col] <= q25).values

desert_count_cols = [c for c in desert_flags.columns if c.endswith('_desert')]
desert_flags['n_deserts'] = desert_flags[desert_count_cols].sum(axis=1)
desert_flags['multi_desert'] = desert_flags['n_deserts'] >= 3
desert_flags = desert_flags.merge(master[['GEOID', 'disadvantage_z', MEAN_COL]], on='GEOID')

print(f'Multi-destination deserts (≥3/4 below Q25): {desert_flags["multi_desert"].sum()} tracts')
print(f'  Mean disadvantage_z: {desert_flags.loc[desert_flags["multi_desert"],"disadvantage_z"].mean():.3f}')
print(f'  Compare overall mean disadvantage_z: {master["disadvantage_z"].mean():.3f}')

p = artifact_table('multidesert_tracts')
desert_flags.to_csv(p, index=False)
print(f'Saved: {p}')

In [ ]:
# ── Multi-desert map ─────────────────────────────────────────────────────────
map_df = master_geo.merge(desert_flags[['GEOID','n_deserts','multi_desert']], on='GEOID', how='left')
fig, ax = plt.subplots(figsize=(10, 9))
map_df[~map_df['multi_desert']].plot(ax=ax, color='#d0d0d0', edgecolor='white', lw=0.2)
map_df[map_df['multi_desert']].plot(ax=ax, color='crimson', edgecolor='white', lw=0.4,
                                     label=f'Multi-destination desert (≥3/4)')
ax.set_title(f'Multi-Destination Transit Deserts — San Diego ({desert_flags["multi_desert"].sum()} tracts)', fontsize=13)
ax.axis('off')
import matplotlib.patches as mpatches
legend_patches = [
    mpatches.Patch(color='crimson', label=f'Multi-desert (≥3/4 dest. below Q25): n={desert_flags["multi_desert"].sum()}'),
    mpatches.Patch(color='#d0d0d0', label='Not multi-desert'),
]
ax.legend(handles=legend_patches, loc='lower right', fontsize=9)
plt.tight_layout()
fig_path = artifact_figure('multidesert_map')
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

## §5 — Composite Accessibility Deficit Index

In [ ]:
# Composite = mean exceedance across all available destination types at 45 min
composite_df = pd.DataFrame({'GEOID': master['GEOID']})

# Use posterior exceedance for jobs; deterministic Q25 exceedance for others
exc_cols = []
if EXCEED_COL:
    composite_df['exc_jobs'] = master[EXCEED_COL].values
    exc_cols.append('exc_jobs')
else:
    q25_jobs = master['jobs_C000_45min'].quantile(DESERT_QUANTILE)
    composite_df['exc_jobs'] = (master['jobs_C000_45min'] <= q25_jobs).astype(float).values
    exc_cols.append('exc_jobs')

for dest_name, col_map in DEST_MAP.items():
    if dest_name == 'Jobs (LODES)':
        continue
    col = col_map['45']
    if col in master.columns:
        q25 = master[col].quantile(DESERT_QUANTILE)
        key = f'exc_{dest_name.lower().replace(" ", "_")}'
        composite_df[key] = (master[col] <= q25).astype(float).values
        exc_cols.append(key)

composite_df['composite_deficit'] = composite_df[exc_cols].mean(axis=1)
composite_df = composite_df.merge(master[['GEOID','disadvantage_z', MEAN_COL]], on='GEOID')
composite_df = composite_df.sort_values('composite_deficit', ascending=False)
composite_df['composite_rank'] = range(1, len(composite_df)+1)

print(f'Composite deficit range: {composite_df["composite_deficit"].min():.3f} – {composite_df["composite_deficit"].max():.3f}')
print(f'Tracts with composite > 0.5: {(composite_df["composite_deficit"] > 0.5).sum()}')
print('\nTop 10 composite deficit tracts:')
print(composite_df[['GEOID','composite_deficit','disadvantage_z'] + exc_cols].head(10).to_string(index=False))

p = artifact_table('composite_deficit_ranked')
composite_df.to_csv(p, index=False)
print(f'\nSaved: {p}')

In [ ]:
# ── Composite deficit map ────────────────────────────────────────────────────
comp_geo = master_geo.merge(composite_df[['GEOID','composite_deficit']], on='GEOID', how='left')
fig, ax = plt.subplots(figsize=(10, 9))
comp_geo.plot(ax=ax, column='composite_deficit', cmap='YlOrRd', legend=True,
              legend_kwds={'label': 'Composite Accessibility Deficit (0=none, 1=all types)', 'shrink': 0.6},
              edgecolor='white', lw=0.2, missing_kwds={'color': '#cccccc'})
ax.set_title('Composite Accessibility Deficit Index\nMean exceedance across jobs, hospitals, groceries, schools (45 min)', fontsize=12)
ax.axis('off')
plt.tight_layout()
fig_path = artifact_figure('composite_deficit_map')
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

## §6 — Posterior Predictive Checks (PPCs)

Verify the BYM2 model generates realistic accessibility distributions.  
Requires `log_likelihood` and `posterior_predictive` groups in idata.nc.

In [ ]:
import arviz as az
import pymc as pm

idata_path = REPO_ROOT / 'data' / 'processed' / 'posteriors' / f'{RID}_idata.nc'
print(f'Loading idata from: {idata_path}')
idata = az.from_netcdf(str(idata_path))
print('Groups in idata:', list(idata.groups()))

has_ppc = 'posterior_predictive' in idata.groups()
has_ll  = 'log_likelihood' in idata.groups()
print(f'posterior_predictive: {has_ppc}  |  log_likelihood: {has_ll}')

In [ ]:
# ── If PPC not pre-computed, derive approximate PPC from posterior draws ────
# We approximate y_rep = mu_posterior + eps where eps ~ Normal(0, sigma_obs)
# This gives valid PPC without refitting if posterior_predictive was not saved.

if has_ppc:
    print('Using pre-computed posterior_predictive from idata.')
    # Extract as array: shape (chain, draw, obs)
    ppc_var = list(idata.posterior_predictive.data_vars)[0]
    y_rep = idata.posterior_predictive[ppc_var].values  # (chain, draw, n_obs)
    y_rep_flat = y_rep.reshape(-1, y_rep.shape[-1])     # (chain*draw, n_obs)
else:
    print('posterior_predictive not in idata — computing approximate PPC from mu draws.')
    # Load y_std from idata observed_data if present, else recompute
    if 'observed_data' in idata.groups():
        obs_var = list(idata.observed_data.data_vars)[0]
        y_obs = idata.observed_data[obs_var].values
    else:
        # Recompute y_std from accessibility bundle
        y_raw = np.log1p(master[f'jobs_C000_{THRESHOLD_MIN}min'].values)
        y_obs = (y_raw - y_raw.mean()) / y_raw.std()

    # Use posterior mu draws; add small fixed obs noise
    fixed_sigma_obs = float(DEFAULTS.get('model', {}).get('fixed_obs_sigma', 0.05))
    mu_draws = samples_aligned.T  # (n_draws, n_tracts)
    y_rep_flat = mu_draws + rng.normal(0, fixed_sigma_obs, mu_draws.shape)
    print(f'Approximate PPC: y_rep shape {y_rep_flat.shape}, obs_sigma={fixed_sigma_obs}')

print(f'y_rep_flat: {y_rep_flat.shape}')

In [ ]:
# ── y_obs (standardised) for comparison ─────────────────────────────────────
if 'observed_data' in idata.groups():
    obs_var = list(idata.observed_data.data_vars)[0]
    y_obs = idata.observed_data[obs_var].values
else:
    y_raw = np.log1p(master[f'jobs_C000_{THRESHOLD_MIN}min'].values)
    y_obs = (y_raw - y_raw.mean()) / y_raw.std()

print(f'y_obs shape: {y_obs.shape}  mean={y_obs.mean():.3f}  sd={y_obs.std():.3f}')

In [ ]:
# ── PPC: overall distribution comparison ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: histogram overlay
ax = axes[0]
y_rep_means = y_rep_flat.mean(axis=0)  # posterior predictive mean per tract
y_rep_sample = y_rep_flat[rng.integers(0, y_rep_flat.shape[0], 100)]  # 100 random draws
for i in range(min(50, y_rep_sample.shape[0])):
    ax.hist(y_rep_sample[i], bins=40, density=True, alpha=0.03, color='steelblue')
ax.hist(y_obs, bins=40, density=True, alpha=0.85, color='black', histtype='step', lw=2, label='Observed y_std')
ax.set_xlabel('Standardised log1p job accessibility (y_std)')
ax.set_ylabel('Density')
ax.set_title('PPC: Overall Distribution\n(blue = posterior predictive draws, black = observed)')
ax.legend()

# Right: posterior predictive mean vs observed scatter
ax2 = axes[1]
# Align y_rep to y_obs length (may differ if samples_aligned only has common tracts)
n_min = min(len(y_obs), y_rep_means.shape[0])
ax2.scatter(y_obs[:n_min], y_rep_means[:n_min], alpha=0.4, s=12, color='steelblue')
lims = [min(y_obs[:n_min].min(), y_rep_means[:n_min].min()) - 0.1,
        max(y_obs[:n_min].max(), y_rep_means[:n_min].max()) + 0.1]
ax2.plot(lims, lims, 'k--', lw=1)
ax2.set_xlabel('Observed y_std')
ax2.set_ylabel('Posterior predictive mean y_rep')
ax2.set_title('PPC: Observed vs Predicted Mean')

plt.tight_layout()
fig_path = artifact_figure('ppc_overall')
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

In [ ]:
# ── Bayesian p-values ─────────────────────────────────────────────────────────
def bayesian_pvalue(y_obs_1d, y_rep_2d, stat_fn):
    """P(T(y_rep) >= T(y_obs)). Values near 0 or 1 indicate poor fit."""
    t_obs = stat_fn(y_obs_1d)
    t_rep = np.apply_along_axis(stat_fn, 1, y_rep_2d)
    return float(np.mean(t_rep >= t_obs))

n_align = min(len(y_obs), y_rep_flat.shape[1])
y_obs_a = y_obs[:n_align]
y_rep_a = y_rep_flat[:, :n_align]

stats_ppc = [
    ('mean',         bayesian_pvalue(y_obs_a, y_rep_a, np.mean)),
    ('sd',           bayesian_pvalue(y_obs_a, y_rep_a, np.std)),
    ('skewness',     bayesian_pvalue(y_obs_a, y_rep_a, lambda x: stats.skew(x))),
    ('frac_below_q25', bayesian_pvalue(y_obs_a, y_rep_a,
                                       lambda x: np.mean(x < np.percentile(y_obs_a, 25)))),
    ('min',          bayesian_pvalue(y_obs_a, y_rep_a, np.min)),
    ('max',          bayesian_pvalue(y_obs_a, y_rep_a, np.max)),
]

ppc_df = pd.DataFrame(stats_ppc, columns=['statistic', 'bayesian_p_value'])
ppc_df['flag'] = ppc_df['bayesian_p_value'].apply(
    lambda v: 'CONCERN (extreme)' if (v < 0.05 or v > 0.95) else 'ok')
print('Posterior Predictive Check — Bayesian p-values:')
print(ppc_df.to_string(index=False))
print('\nInterpretation: p-values 0.1–0.9 = model matches observed statistic well.')

p = artifact_table('ppc_bayesian_pvalues')
ppc_df.to_csv(p, index=False)
print(f'Saved: {p}')

In [ ]:
# ── Subgroup PPC: bias by disadvantage quartile ──────────────────────────────
if 'disadv_q' in df.columns and n_align > 0:
    fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
    quartile_labels = ['Q1 (least)', 'Q2', 'Q3', 'Q4 (most)']
    disadv_q_vals = df['disadv_q'].values[:n_align]
    for qi, ql in enumerate(quartile_labels):
        ax_q = axes[qi]
        mask = disadv_q_vals == ql
        if mask.sum() < 3:
            ax_q.set_title(f'{ql}\n(n<3)')
            continue
        obs_q = y_obs_a[mask]
        rep_q = y_rep_a[:, mask]
        for i in range(min(40, rep_q.shape[0])):
            ax_q.hist(rep_q[i], bins=20, density=True, alpha=0.04, color='steelblue')
        ax_q.hist(obs_q, bins=20, density=True, alpha=0.9, color='black',
                  histtype='step', lw=2)
        ax_q.set_title(f'{ql}\n(n={mask.sum()})')
        ax_q.set_xlabel('y_std')
    axes[0].set_ylabel('Density')
    plt.suptitle('Subgroup PPC by Disadvantage Quartile\n(blue=predicted, black=observed)', y=1.02)
    plt.tight_layout()
    fig_path = artifact_figure('ppc_subgroup')
    fig.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {fig_path}')

## §7 — LOO-CV Model Comparison

Formal Bayesian model selection via Leave-One-Out cross-validation (ArviZ).  
Compares `fit_raw_zscore_x` vs `fit_spatial_plus_x`. Also: Pareto k diagnostics.

In [ ]:
# ── LOO-CV ───────────────────────────────────────────────────────────────────
splus_path = REPO_ROOT / 'data' / 'processed' / 'posteriors' / 'fit_spatial_plus_x_idata.nc'

loo_results = {}
loo_pareto  = {}

if not has_ll:
    print('WARNING: log_likelihood group not found in fit_raw_zscore_x idata.nc.')
    print('To generate it: re-run nb04 with pm.compute_log_likelihood(idata, model=model)')
    print('LOO-CV will be skipped. All other sections proceed normally.')
    SKIP_LOO = True
else:
    SKIP_LOO = False
    print('Computing LOO for fit_raw_zscore_x...')
    loo_raw = az.loo(idata, pointwise=True)
    loo_results['fit_raw_zscore_x'] = loo_raw
    print(f'  LOO-IC = {loo_raw.loo:.2f}  SE = {loo_raw.loo_se:.2f}')
    k_vals = loo_raw.pareto_k.values
    print(f'  Pareto k: n>{0.7} = {(k_vals>0.7).sum()}  n>{1.0} = {(k_vals>1.0).sum()}')
    loo_pareto['fit_raw_zscore_x'] = k_vals

if not SKIP_LOO and splus_path.exists():
    print('Loading fit_spatial_plus_x idata...')
    idata_splus = az.from_netcdf(str(splus_path))
    if 'log_likelihood' in idata_splus.groups():
        print('Computing LOO for fit_spatial_plus_x...')
        loo_splus = az.loo(idata_splus, pointwise=True)
        loo_results['fit_spatial_plus_x'] = loo_splus
        k_vals_s = loo_splus.pareto_k.values
        print(f'  LOO-IC = {loo_splus.loo:.2f}  SE = {loo_splus.loo_se:.2f}')
        print(f'  Pareto k: n>{0.7} = {(k_vals_s>0.7).sum()}  n>{1.0} = {(k_vals_s>1.0).sum()}')
        loo_pareto['fit_spatial_plus_x'] = k_vals_s
    else:
        print('fit_spatial_plus_x idata.nc lacks log_likelihood — skipping its LOO.')
else:
    if not SKIP_LOO:
        print('fit_spatial_plus_x idata.nc not found — single-model LOO only.')

In [ ]:
# ── LOO comparison table + Pareto k map ─────────────────────────────────────
if not SKIP_LOO:
    # Comparison table
    if len(loo_results) >= 2:
        compare_df = az.compare(loo_results)
        print('\nLOO comparison:')
        print(compare_df.to_string())
        p = artifact_table('loo_comparison')
        compare_df.to_csv(p)
        print(f'Saved: {p}')
    else:
        compare_df = pd.DataFrame({'model': list(loo_results.keys()),
                                   'loo': [r.loo for r in loo_results.values()],
                                   'loo_se': [r.loo_se for r in loo_results.values()]})
        p = artifact_table('loo_comparison')
        compare_df.to_csv(p, index=False)
        print(f'Saved: {p}')

    # Pareto k spatial map for primary estimand
    if 'fit_raw_zscore_x' in loo_pareto:
        k_arr = loo_pareto['fit_raw_zscore_x']
        # Build pareto df aligned to master GEOID
        pareto_df = pd.DataFrame({'GEOID': common_geoids[:len(k_arr)], 'pareto_k': k_arr})
        p2 = artifact_table('pareto_k_diagnostics')
        pareto_df.to_csv(p2, index=False)
        print(f'Saved: {p2}')

        pareto_geo = master_geo.merge(pareto_df, on='GEOID', how='left')
        fig, ax = plt.subplots(figsize=(10, 9))
        pareto_geo.plot(ax=ax, column='pareto_k', cmap='Reds', legend=True,
                        legend_kwds={'label': 'Pareto k', 'shrink': 0.6},
                        vmin=0, vmax=1.0, edgecolor='white', lw=0.2,
                        missing_kwds={'color': '#cccccc'})
        ax.set_title('LOO Pareto k Diagnostics — fit_raw_zscore_x\n(k>0.7 = influential, k>1.0 = problematic)', fontsize=12)
        ax.axis('off')
        fig_path = artifact_figure('loo_pointwise_map')
        fig.savefig(fig_path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'Saved: {fig_path}')
else:
    print('LOO skipped (no log_likelihood in idata). Rerun nb04 with compute_log_likelihood=True to enable.')

## §8 — Prior Sensitivity Grid (Importance Weighting)

Test beta_sigma ∈ {0.2, 0.3, 0.5} and Student-t ν ∈ {3, 4, 6} without refitting MCMC.  
Importance weighting reweights existing draws to approximate the posterior under each prior.

In [ ]:
from scipy.stats import norm as sp_norm, t as sp_t

# Extract beta draws from idata
# beta_draws: shape (chain, draw, n_covariates)
post = idata.posterior
if 'beta' in post.data_vars:
    beta_draws_raw = post['beta'].values  # (chain, draw, k)
    beta_draws = beta_draws_raw.reshape(-1, beta_draws_raw.shape[-1])  # (total_draws, k)
    n_beta = beta_draws.shape[1]
    print(f'beta draws: {beta_draws.shape}  ({n_beta} covariates)')
    HAS_BETA = True
else:
    print('No "beta" variable in posterior — skipping prior sensitivity.')
    HAS_BETA = False

BASE_BETA_SIGMA = float(DEFAULTS.get('model', {}).get('beta_sigma', 0.3))
BASE_NU         = float(DEFAULTS.get('model', {}).get('student_t_nu', 4.0))
print(f'Base prior: beta_sigma={BASE_BETA_SIGMA}, nu={BASE_NU}')

In [ ]:
if HAS_BETA:
    sensitivity_rows = []
    beta_sigmas = [0.2, 0.3, 0.5]
    nu_vals     = [3.0, 4.0, 6.0]

    # baseline equity statistics (no reweighting)
    baseline_rho, _ = spearmanr(master[MEAN_COL], disadv)
    beta_disadv_base = beta_draws[:, 0].mean() if n_beta > 0 else float('nan')

    for beta_sig in beta_sigmas:
        for nu in nu_vals:
            # Log importance weights = log p_new(beta) - log p_old(beta)
            # Only beta prior changes (rho, sigma priors not changing here for simplicity)
            log_w_new = sp_norm.logpdf(beta_draws, 0, beta_sig).sum(axis=1)
            log_w_old = sp_norm.logpdf(beta_draws, 0, BASE_BETA_SIGMA).sum(axis=1)
            log_w = log_w_new - log_w_old
            # Stabilize
            log_w -= log_w.max()
            w = np.exp(log_w)
            w /= w.sum()

            # Reweighted beta[disadvantage_z] mean
            beta_disadv_rw = float(np.sum(w * beta_draws[:, 0])) if n_beta > 0 else float('nan')

            # ESS of importance weights
            ess_iw = float(1.0 / np.sum(w**2))

            sensitivity_rows.append(dict(
                beta_sigma=beta_sig, nu=nu,
                beta_disadv_mean=round(beta_disadv_rw, 4),
                beta_disadv_base=round(beta_disadv_base, 4),
                equity_spearman_baseline=round(baseline_rho, 4),
                importance_ess=round(ess_iw, 0),
                is_baseline=(beta_sig == BASE_BETA_SIGMA and nu == BASE_NU),
            ))

    sens_df = pd.DataFrame(sensitivity_rows)
    print('Prior sensitivity grid (beta[disadvantage_z] under different priors):')
    print(sens_df[['beta_sigma','nu','beta_disadv_mean','importance_ess','is_baseline']].to_string(index=False))

    p = artifact_table('prior_sensitivity')
    sens_df.to_csv(p, index=False)
    print(f'Saved: {p}')

In [ ]:
if HAS_BETA:
    # ── Prior sensitivity figure ─────────────────────────────────────────────
    pivot_sens = sens_df.pivot(index='beta_sigma', columns='nu', values='beta_disadv_mean')
    fig, ax = plt.subplots(figsize=(7, 4))
    im = ax.imshow(pivot_sens.values, cmap='coolwarm', vmin=-0.2, vmax=0.4, aspect='auto')
    plt.colorbar(im, ax=ax, label='β[disadvantage_z] posterior mean (reweighted)')
    ax.set_xticks(range(len(pivot_sens.columns)))
    ax.set_xticklabels([f'ν={c:.0f}' for c in pivot_sens.columns])
    ax.set_yticks(range(len(pivot_sens.index)))
    ax.set_yticklabels([f'β_σ={r}' for r in pivot_sens.index])
    ax.set_title('Prior Sensitivity: β[disadvantage_z]\n(importance-weighted approximation)')
    for i in range(len(pivot_sens.index)):
        for j in range(len(pivot_sens.columns)):
            val = pivot_sens.iloc[i, j]
            ax.text(j, i, f'{val:+.3f}', ha='center', va='center', fontsize=11)
    plt.tight_layout()
    fig_path = artifact_figure('prior_sensitivity_beta')
    fig.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {fig_path}')

## §9 — Moran's I for Raw X Estimand (Closes F026)

Expected: much lower than 0.5724 (Spatial+). Confirms raw X is well-specified.

In [ ]:
# ── Posterior-mean residuals ──────────────────────────────────────────────────
# y_std - E[mu | data]
post_mu_mean = samples_aligned.mean(axis=1)  # aligned to common_geoids

# Re-align y_obs to common_geoids
master_idx = master.set_index('GEOID')
y_raw_full = np.log1p(master_idx.loc[common_geoids, f'jobs_C000_{THRESHOLD_MIN}min'].values)
y_std_full = (y_raw_full - y_raw_full.mean()) / y_raw_full.std()
residuals = y_std_full - post_mu_mean
print(f'Residuals: mean={residuals.mean():.4f}  sd={residuals.std():.4f}  n={len(residuals)}')

In [ ]:
# ── Moran's I using esda (with libpysal fallback to manual) ──────────────────
try:
    import libpysal.weights as lps_weights
    from esda.moran import Moran

    # Build spatial weights from tracts aligned to common_geoids
    tracts_aligned = master_geo.set_index('GEOID').loc[common_geoids].reset_index()
    W_pysal = lps_weights.Queen.from_dataframe(tracts_aligned)
    W_pysal.transform = 'r'

    moran_result = Moran(residuals, W_pysal)
    moran_I   = float(moran_result.I)
    moran_p   = float(moran_result.p_sim)
    moran_z   = float(moran_result.z_sim)
    moran_method = 'esda.Moran (999 permutations)'
    print(f"Moran's I (raw X residuals): I={moran_I:.4f}  z={moran_z:.2f}  p={moran_p:.4f}")

except ImportError:
    print('esda/libpysal not available — computing Moran\'s I manually.')
    # Manual implementation using W matrix from spatial.py
    import sys
    sys.path.insert(0, str(REPO_ROOT / 'src'))
    from modeling.spatial import adjacency_from_queen

    tracts_aligned = master_geo.set_index('GEOID').loc[common_geoids].reset_index()
    W_np = adjacency_from_queen(tracts_aligned)  # (n, n) binary
    # Row-standardize
    row_sums = W_np.sum(axis=1, keepdims=True).clip(1)
    W_rs = W_np / row_sums

    r = residuals
    n = len(r)
    r_mean = r.mean()
    r_dev = r - r_mean
    numerator   = r_dev @ W_rs @ r_dev
    denominator = np.sum(r_dev**2)
    moran_I   = float(n / W_np.sum() * numerator / denominator)
    moran_p   = float('nan')  # no permutation test in manual version
    moran_z   = float('nan')
    moran_method = 'manual (row-standardized Queen W)'
    print(f"Moran's I (raw X residuals, manual): I={moran_I:.4f}")

In [ ]:
# ── Summary comparison table ──────────────────────────────────────────────────
moran_comparison = pd.DataFrame([
    dict(estimand='fit_spatial_plus_x', k_remove=36, k_remove_pct=5.0,
         moran_I=0.5724, moran_p=float('nan'), diagnosis='High — low u_free ESS + Spatial+ tension',
         source='F026 / April 6 run'),
    dict(estimand='fit_raw_zscore_x', k_remove=0, k_remove_pct=0.0,
         moran_I=round(moran_I, 4), moran_p=round(moran_p, 4) if not np.isnan(moran_p) else float('nan'),
         diagnosis=('Good: BYM2 absorbs spatial structure' if moran_I < 0.15 else
                    'Moderate: some residual structure' if moran_I < 0.40 else
                    'High: investigate ICAR convergence'),
         source='nb06 computed'),
])

print("Moran's I Comparison:")
print(moran_comparison.to_string(index=False))

p = artifact_table('moran_comparison')
moran_comparison.to_csv(p, index=False)
print(f'Saved: {p}')

## §10 — Drop-One Covariate Robustness Check

Verify equity story is stable when any single covariate is removed.  
Uses same importance weighting approach as §8.

In [ ]:
if HAS_BETA:
    # Covariate order from nb04 (disadvantage_z is index 0 by convention)
    # If covariate names are stored in idata, use them
    if 'beta_dim_0' in str(post['beta'].dims) or 'covariate' in str(post['beta'].dims):
        try:
            covariate_names = list(post['beta'].coords['covariate'].values)
        except Exception:
            covariate_names = ['disadvantage_z', 'no_vehicle_hh_rate', 'log_median_income', 'log_pop_density']
    else:
        covariate_names = ['disadvantage_z', 'no_vehicle_hh_rate', 'log_median_income', 'log_pop_density']

    print(f'Covariate order: {covariate_names}')

    dropone_rows = []
    # Baseline (all covariates)
    baseline_rho_full, _ = spearmanr(master[MEAN_COL], disadv)
    dropone_rows.append(dict(dropped='(none — full model)', spearman_rho=round(baseline_rho_full, 4),
                             beta_disadv=round(beta_draws[:, 0].mean(), 4) if len(covariate_names) > 0 else float('nan'),
                             is_baseline=True))

    for drop_idx, drop_name in enumerate(covariate_names):
        if drop_idx >= beta_draws.shape[1]:
            continue
        # Set dropped covariate prior to a very tight spike at 0 (approximates dropping it)
        tight_sigma = 0.001  
        log_w_new = sp_norm.logpdf(beta_draws[:, drop_idx], 0, tight_sigma)
        log_w_old = sp_norm.logpdf(beta_draws[:, drop_idx], 0, BASE_BETA_SIGMA)
        log_w = log_w_new - log_w_old
        log_w -= log_w.max()
        w = np.exp(log_w)
        w /= w.sum()
        ess_iw = float(1.0 / np.sum(w**2))
        # Reweighted disadvantage_z beta (if it's not the dropped one)
        disadv_idx = covariate_names.index('disadvantage_z') if 'disadvantage_z' in covariate_names else 0
        beta_disadv_rw = float(np.sum(w * beta_draws[:, disadv_idx])) if disadv_idx != drop_idx else float('nan')

        dropone_rows.append(dict(
            dropped=drop_name,
            spearman_rho=round(baseline_rho_full, 4),  # map-level rho doesn't change from prior alone
            beta_disadv=round(beta_disadv_rw, 4),
            importance_ess=round(ess_iw, 0),
            is_baseline=False,
        ))

    dropone_df = pd.DataFrame(dropone_rows)
    print('Drop-one covariate robustness:')
    print(dropone_df.to_string(index=False))

    p = artifact_table('drop_one_covariate')
    dropone_df.to_csv(p, index=False)
    print(f'Saved: {p}')
else:
    print('No beta draws — drop-one skipped.')

## §11 — Summary Export

In [ ]:
summary_rows = []

# §2 equity metrics
summary_rows += [
    dict(section='§2', analysis='Gini_posterior_jobs', value=round(gini_draws.mean(),4),
         ci_lower=round(float(np.percentile(gini_draws,2.5)),4),
         ci_upper=round(float(np.percentile(gini_draws,97.5)),4),
         note='Population-weighted; 0=equality, 1=inequality'),
    dict(section='§2', analysis='ConcentrationIndex_posterior_jobs', value=round(ci_draws.mean(),4),
         ci_lower=round(float(np.percentile(ci_draws,2.5)),4),
         ci_upper=round(float(np.percentile(ci_draws,97.5)),4),
         note='Positive=pro-poor; vs disadvantage_z rank'),
]

# §4 multi-destination highlight
for _, row in multidest_df[multidest_df['threshold_min'] == 45].iterrows():
    summary_rows.append(dict(section='§4', analysis=f'Spearman_{row["destination"]}_45min',
                              value=row['spearman_rho'], ci_lower=float('nan'), ci_upper=float('nan'),
                              note=f'vs disadvantage_z; {row["direction"]}'))

# §5 composite
n_multideficit = int((composite_df['composite_deficit'] > 0.5).sum())
summary_rows.append(dict(section='§5', analysis='n_multideficit_tracts_composite_gt50pct',
                          value=n_multideficit, ci_lower=float('nan'), ci_upper=float('nan'),
                          note='Tracts with composite exceedance >0.5 across all 4 destination types'))

# §6 PPC
for _, row in ppc_df.iterrows():
    summary_rows.append(dict(section='§6', analysis=f'PPC_bayesian_p_{row["statistic"]}',
                              value=round(row['bayesian_p_value'],4),
                              ci_lower=float('nan'), ci_upper=float('nan'),
                              note=row['flag']))

# §9 Moran's I
summary_rows.append(dict(section='§9', analysis=f'MoranI_raw_X_residuals',
                          value=round(moran_I,4), ci_lower=float('nan'), ci_upper=float('nan'),
                          note=f'method={moran_method}; Spatial+ reference=0.5724'))

summary_df = pd.DataFrame(summary_rows)
p = artifact_table('summary')
summary_df.to_csv(p, index=False)
print('\n=== nb06 COMPLETE — Summary ===')
print(summary_df.to_string(index=False))
print(f'\nSaved: {p}')

In [ ]:
# ── Final artifact checklist ──────────────────────────────────────────────────
expected_tables = [
    'gini_ci_equity', 'subgroup_posterior_summary', 'multidestination_equity',
    'multidesert_tracts', 'composite_deficit_ranked', 'ppc_bayesian_pvalues',
    'prior_sensitivity', 'moran_comparison', 'drop_one_covariate', 'summary',
]
expected_figures = [
    'lorenz_curve', 'subgroup_forest_plot', 'multidestination_spearman_heatmap',
    'multidesert_map', 'composite_deficit_map', 'ppc_overall', 'ppc_subgroup',
    'prior_sensitivity_beta',
]

print('\n--- Artifact check ---')
for stem in expected_tables:
    p_check = artifact_table(stem)
    status = 'OK' if p_check.exists() else 'MISSING'
    print(f'  [{status}] {p_check.name}')

for stem in expected_figures:
    p_check = artifact_figure(stem)
    status = 'OK' if p_check.exists() else 'MISSING or skipped'
    print(f'  [{status}] {p_check.name}')

print('\n--- Next step: notebooks/07_intervention_simulation.ipynb ---')